In [2]:
# ============================================================
# ERIP - ENTERPRISE RISK INTELLIGENCE PLATFORM
# ============================================================
#
# Notebook
# --------
# nb_build_customer360
#
# Layer
# -----
# Silver Layer
#
# Purpose
# -------
# Build the integrated Customer 360 analytical entity by
# combining Customer, Loan and Internal Rating dimensions.
#
# Business Objective
# ------------------
# Create a single customer-centric analytical dataset used by:
#
# • Executive Dashboards
# • Customer 360
# • Credit Portfolio Analytics
# • Expected Credit Loss
# • Relationship Management
# • AI Decision Intelligence
#
# Enterprise Concepts
# -------------------
# ✓ Customer 360
# ✓ Enterprise Data Integration
# ✓ Credit Risk Analytics
# ✓ Dimensional Modelling
# ✓ Medallion Architecture
# ============================================================

from pyspark.sql.functions import *
from datetime import datetime

# ------------------------------------------------------------
# Source / Target Configuration
# ------------------------------------------------------------

customer_table = "silver_customer"
loan_table = "silver_loan"
rating_table = "silver_rating"

target_table = "silver_customer360"

pipeline_name = "nb_build_customer360"

run_start_time = datetime.now()

print("ERIP Customer360 Build Started")

StatementMeta(, 62309bcc-02ae-4275-ae4d-a165da1eb437, 4, Finished, Available, Finished, False)

ERIP Customer360 Build Started


In [3]:
# ============================================================
# SECTION 2 - READ SILVER TABLES
# ============================================================
#
# Purpose
# -------
# Read the Silver business entities that will be integrated
# into Customer360.
#
# Enterprise Concepts
# -------------------
# ✓ Enterprise Data Integration
# ✓ Delta Lake
# ✓ Data Lineage
# ============================================================

customer_df = spark.table(customer_table)

loan_df = spark.table(loan_table)

rating_df = spark.table(rating_table)

print(f"Customers : {customer_df.count()}")
print(f"Loans     : {loan_df.count()}")
print(f"Ratings   : {rating_df.count()}")

StatementMeta(, 62309bcc-02ae-4275-ae4d-a165da1eb437, 5, Finished, Available, Finished, False)

Customers : 1000
Loans     : 5000
Ratings   : 1000


In [4]:
# ============================================================
# SECTION 3 - BUILD CUSTOMER LOAN SUMMARY
# ============================================================

loan_summary = (

    loan_df

    .groupBy("customer_id")

    .agg(

        count("loan_id").alias("loan_count"),

        sum("approved_limit").alias("total_limit"),

        sum("outstanding_balance").alias("total_outstanding"),

        sum("exposure_at_default").alias("total_ead"),

        avg("interest_rate_pct").alias("average_interest_rate")

    )

)

display(loan_summary.limit(10))

StatementMeta(, 62309bcc-02ae-4275-ae4d-a165da1eb437, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 149a14d1-acfc-442d-962d-88d700e4db2d)

In [5]:
# ============================================================
# SECTION 4 - BUILD CUSTOMER RATING SUMMARY
# ============================================================

rating_summary = (

    rating_df

    .groupBy("customer_id")

    .agg(

        avg("pd").alias("average_pd"),

        avg("lgd").alias("average_lgd"),

        avg("expected_loss").alias("average_expected_loss"),

        max("ifrs9_stage_numeric").alias("highest_ifrs9_stage"),

        max("is_watchlist").alias("watchlist_flag")

    )

)

display(rating_summary.limit(10))

StatementMeta(, 62309bcc-02ae-4275-ae4d-a165da1eb437, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 906caff3-1158-4fb7-b6fa-fe36062b2c22)

In [6]:
# ============================================================
# SECTION 5 - BUILD CUSTOMER360
# ============================================================
#
# Purpose
# -------
# Join Customer, Loan and Rating entities into a unified
# customer-centric analytical dataset.
# ============================================================

customer360_df = (

    customer_df.alias("c")

    .join(

        loan_summary.alias("l"),

        "customer_id",

        "left"

    )

    .join(

        rating_summary.alias("r"),

        "customer_id",

        "left"

    )

)

StatementMeta(, 62309bcc-02ae-4275-ae4d-a165da1eb437, 8, Finished, Available, Finished, False)

In [7]:
# ============================================================
# SECTION 6 - BUSINESS ENRICHMENT
# ============================================================
#
# Derived Attributes
# ------------------
# • Exposure Band
# • Risk Category
# • Customer Value Segment
# • Relationship Status
# ============================================================

customer360_df = (

    customer360_df

    .withColumn(

        "customer_value_segment",

        when(col("total_ead") >= 50000000, "Strategic")

        .when(col("total_ead") >= 10000000, "Corporate")

        .when(col("total_ead") >= 1000000, "Commercial")

        .otherwise("SME")

    )

    .withColumn(

        "portfolio_risk",

        when(col("average_pd") >= 0.20, "Very High")

        .when(col("average_pd") >= 0.10, "High")

        .when(col("average_pd") >= 0.05, "Medium")

        .otherwise("Low")

    )

    .withColumn(

        "relationship_status",

        when(col("watchlist_flag") == 1, "Watchlist")

        .otherwise("Healthy")

    )

)

display(customer360_df.limit(10))

StatementMeta(, 62309bcc-02ae-4275-ae4d-a165da1eb437, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5bc86cc8-7030-4fca-ba70-bd8587c1e4cf)

In [8]:
# ============================================================
# SECTION 7 - CUSTOMER360 QUALITY VALIDATION
# ============================================================

rows = customer360_df.count()

duplicate_customers = rows - customer360_df.select("customer_id").distinct().count()

null_customer_ids = customer360_df.filter(

    col("customer_id").isNull()

).count()

print("Customer360 Quality")

print("-------------------")

print(f"Rows : {rows}")

print(f"Duplicate Customers : {duplicate_customers}")

print(f"Null Customer IDs : {null_customer_ids}")

if duplicate_customers > 0 or null_customer_ids > 0:

    raise Exception("Customer360 Validation Failed")

else:

    print("✓ Customer360 Validation Passed")

StatementMeta(, 62309bcc-02ae-4275-ae4d-a165da1eb437, 10, Finished, Available, Finished, False)

Customer360 Quality
-------------------
Rows : 1000
Duplicate Customers : 0
Null Customer IDs : 0
✓ Customer360 Validation Passed


In [9]:
# ============================================================
# SECTION 8 - WRITE SILVER CUSTOMER360
# ============================================================

customer360_df.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable(target_table)

print(f"✓ Silver table created : {target_table}")

print(f"Rows written : {customer360_df.count()}")

StatementMeta(, 62309bcc-02ae-4275-ae4d-a165da1eb437, 11, Finished, Available, Finished, False)

✓ Silver table created : silver_customer360
Rows written : 1000
